# Demo Modul 7: Recurrent Neural Network

**Durasi sesi:** 120 menit  
**Kasus:** klasifikasi berita AG News menjadi empat kelas.

Enam modul sebelumnya memakai masukan berukuran tetap. Notebook ini menghadapi panjang yang berbeda antarcontoh — dan jebakan yang muncul karenanya.

## Capaian demo

Setelah demo, praktikan dapat:

1. mengubah teks menjadi tensor melalui tokenisasi, vocabulary, dan bantalan;
2. membangun RNN many-to-one serta membaca bentuk tensornya;
3. **membuktikan** bahwa bantalan merusak representasi akhir bila tidak dikemas;
4. mencatat norma gradien dan menilai pengaruh pemotongan; dan
5. membandingkan RNN dengan pembanding non-sekuensial.

In [ ]:
import platform
import random
import re
import time
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
pd.set_option('display.precision', 4)
print({'torch': torch.__version__, 'device': str(DEVICE)})

## 1. Teks menjadi tensor

Alurnya: teks → token → vocabulary → indeks → tensor `(B, n)`. Vocabulary dibangun **hanya dari subset latih**.

In [ ]:
ROOT = Path('../../data/raw/ag_news')
if not ROOT.exists():
    ROOT = Path('data/raw/ag_news')

kolom = ['label', 'judul', 'ringkasan']
tr = pd.read_csv(ROOT / 'train.csv', names=kolom, header=None)
te = pd.read_csv(ROOT / 'test.csv', names=kolom, header=None)

# Sebagian rilis AG News memakai baris header; buang bila terbaca sebagai teks.
if not str(tr.iloc[0]['label']).strip().isdigit():
    tr, te = tr.iloc[1:].reset_index(drop=True), te.iloc[1:].reset_index(drop=True)

for df in (tr, te):
    df['teks'] = (df['judul'].astype(str) + ' ' + df['ringkasan'].astype(str))
    df['y'] = df['label'].astype(int) - 1        # kelas menjadi 0..3

print(tr.shape, te.shape)
print('label unik:', sorted(tr['y'].unique()))
print(tr[['teks', 'y']].head(2).to_string(index=False))

In [ ]:
idx_latih, idx_val = train_test_split(
    np.arange(len(tr)), train_size=12_000, test_size=3_000,
    stratify=tr['y'].values, random_state=SEED)

teks_latih = tr['teks'].values[idx_latih]
teks_val = tr['teks'].values[idx_val]
y_latih = tr['y'].values[idx_latih]
y_val = tr['y'].values[idx_val]

POLA = re.compile(r"[a-z0-9']+")
tokenisasi = lambda s: POLA.findall(s.lower())

# Vocabulary DARI SUBSET LATIH SAJA.
cacah = Counter(t for s in teks_latih for t in tokenisasi(s))
MIN_FREQ = 2
kosakata = ['<pad>', '<unk>'] + [w for w, n in cacah.most_common() if n >= MIN_FREQ]
stoi = {w: i for i, w in enumerate(kosakata)}
V = len(kosakata)

tak_dikenal = sum(1 for s in teks_val for t in tokenisasi(s) if t not in stoi)
total_token_val = sum(len(tokenisasi(s)) for s in teks_val)
print(f'ukuran vocabulary : {V:,}')
print(f'token <unk> di validasi: {100*tak_dikenal/total_token_val:.2f}%')

In [ ]:
MAKS = 60

def ke_indeks(daftar_teks):
    X = torch.zeros(len(daftar_teks), MAKS, dtype=torch.long)   # 0 = <pad>
    panjang = torch.zeros(len(daftar_teks), dtype=torch.long)
    for i, s in enumerate(daftar_teks):
        tok = [stoi.get(t, 1) for t in tokenisasi(s)][:MAKS]    # 1 = <unk>
        tok = tok or [1]                                        # jangan kosong
        X[i, :len(tok)] = torch.tensor(tok)
        panjang[i] = len(tok)
    return X, panjang

X_latih, L_latih = ke_indeks(teks_latih)
X_val, L_val = ke_indeks(teks_val)
ds_latih = TensorDataset(X_latih, L_latih, torch.tensor(y_latih))
ds_val = TensorDataset(X_val, L_val, torch.tensor(y_val))

print('bentuk X:', tuple(X_latih.shape))
print('panjang: min %d, median %d, maks %d' %
      (L_latih.min(), L_latih.median(), L_latih.max()))
print('contoh indeks:', X_latih[0, :12].tolist())

**Pemeriksaan:** panjang asli disimpan **sebelum** pembantalan. Menghitungnya ulang setelah dibantali jauh lebih repot — dan angka itulah yang dibutuhkan pengemasan.

## 2. Model dan bentuk tensornya

In [ ]:
EMB, HID, KELAS = 100, 128, 4

class ModelRNN(nn.Module):
    def __init__(self, pakai_packing=True):
        super().__init__()
        seed_everything(SEED)
        self.emb = nn.Embedding(V, EMB, padding_idx=0)
        self.rnn = nn.RNN(EMB, HID, batch_first=True)
        self.kepala = nn.Linear(HID, KELAS)
        self.pakai_packing = pakai_packing

    def forward(self, X, panjang):
        E = self.emb(X)                                  # (B, n, EMB)
        if self.pakai_packing:
            kemas = pack_padded_sequence(E, panjang.cpu(), batch_first=True,
                                         enforce_sorted=False)
            _, h_n = self.rnn(kemas)
            wakil = h_n[-1]                              # (B, HID)
        else:
            H, _ = self.rnn(E)
            wakil = H[:, -1, :]                          # SALAH pada batch berbantalan
        return self.kepala(wakil)

class RerataEmbedding(nn.Module):
    """Pembanding kejujuran: tanpa RNN sama sekali."""
    def __init__(self):
        super().__init__()
        seed_everything(SEED)
        self.emb = nn.Embedding(V, EMB, padding_idx=0)
        self.kepala = nn.Linear(EMB, KELAS)

    def forward(self, X, panjang):
        E = self.emb(X).sum(dim=1)
        return self.kepala(E / panjang.unsqueeze(1).to(E.dtype))

m = ModelRNN()
xb, lb, yb = ds_latih[0:4]
with torch.no_grad():
    E = m.emb(xb); H, h_n = m.rnn(E)
print('X   :', tuple(xb.shape))
print('E   :', tuple(E.shape))
print('H   :', tuple(H.shape))
print('h_n :', tuple(h_n.shape))

bagian = {'embedding': sum(p.numel() for p in m.emb.parameters()),
          'rnn': sum(p.numel() for p in m.rnn.parameters()),
          'kepala': sum(p.numel() for p in m.kepala.parameters())}
tot = sum(bagian.values())
for k, v in bagian.items():
    print(f'  {k:>9}: {v:>10,} ({100*v/tot:5.1f}%)')
print(f'  {"total":>9}: {tot:>10,}')

**Temuan:** tabel embedding menyerap sekitar $98\%$ parameter. Sama seperti Modul 5, bagian yang paling banyak dibicarakan bukan bagian yang paling besar — dan ukurannya ditentukan vocabulary, bukan arsitektur.

## 3. Membuktikan pengaruh bantalan

Ini inti modul. Susun satu minibatch berisi contoh pendek dan panjang, lalu bandingkan kedua cara mengambil representasi akhir.

In [ ]:
pendek = int(L_val.argmin())
panjang_ = int(L_val.argmax())
print(f'contoh pendek: {L_val[pendek]} token | contoh panjang: {L_val[panjang_]} token')

Xb = X_val[[pendek, panjang_]]
Lb = L_val[[pendek, panjang_]]

m_kemas, m_polos = ModelRNN(True), ModelRNN(False)
m_polos.load_state_dict(m_kemas.state_dict())        # bobot identik

with torch.no_grad():
    E = m_kemas.emb(Xb)
    _, h_n = m_kemas.rnn(pack_padded_sequence(E, Lb.cpu(), batch_first=True,
                                              enforce_sorted=False))
    wakil_kemas = h_n[-1]
    H, _ = m_polos.rnn(E)
    wakil_polos = H[:, -1, :]

selisih = (wakil_kemas - wakil_polos).abs().max(dim=1).values
print(f'selisih maksimum contoh pendek : {selisih[0]:.4f}')
print(f'selisih maksimum contoh panjang: {selisih[1]:.4f}')

In [ ]:
# Tambah bantalan: perpanjang batch dengan kolom <pad> di kanan.
Xb2 = torch.cat([Xb, torch.zeros(2, 20, dtype=torch.long)], dim=1)

with torch.no_grad():
    E2 = m_kemas.emb(Xb2)
    _, h_n2 = m_kemas.rnn(pack_padded_sequence(E2, Lb.cpu(), batch_first=True,
                                               enforce_sorted=False))
    wakil_kemas2 = h_n2[-1]
    H2, _ = m_polos.rnn(E2)
    wakil_polos2 = H2[:, -1, :]

print(f'h_n berubah setelah bantalan ditambah      : '
      f'{(wakil_kemas2 - wakil_kemas).abs().max():.2e}')
print(f'H[:, -1] berubah setelah bantalan ditambah : '
      f'{(wakil_polos2 - wakil_polos).abs().max():.2e}')

Angka pertama praktis nol; angka kedua tidak. Menambah bantalan sama sekali tidak mengubah isi kalimat, tetapi mengubah representasi `H[:, -1]` — dan tidak ada pesan galat yang muncul. Inilah alasan pengemasan dipakai.

## 4. Empat run terkendali

In [ ]:
BATCH, EPOCH = 64, 5

def loader(ds, batch, acak):
    g = torch.Generator().manual_seed(SEED)
    return DataLoader(ds, batch_size=batch, shuffle=acak, generator=g if acak else None)

@torch.no_grad()
def evaluasi(model, ds):
    model.eval()
    kriteria = nn.CrossEntropyLoss(reduction='sum')
    total_loss, benar = 0.0, 0
    for xb, lb, yb in loader(ds, 256, False):
        xb, lb, yb = xb.to(DEVICE), lb, yb.to(DEVICE)
        logits = model(xb, lb)
        total_loss += kriteria(logits, yb).item()
        benar += (logits.argmax(1) == yb).sum().item()
    return total_loss / len(ds), benar / len(ds)

def jalankan(model, label, clip=None, epoch=EPOCH):
    model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    kriteria = nn.CrossEntropyLoss()
    dl = loader(ds_latih, BATCH, True)
    riwayat = {'val_acc': [], 'val_loss': [], 'grad_norm': []}
    n_update, mulai = 0, time.perf_counter()

    for _ in range(epoch):
        model.train()
        for xb, lb, yb in dl:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            kriteria(model(xb, lb), yb).backward()
            # norma dicatat SEBELUM pemotongan
            norma = torch.sqrt(sum((p.grad ** 2).sum() for p in model.parameters()
                                   if p.grad is not None)).item()
            riwayat['grad_norm'].append(norma)
            if clip is not None:
                nn.utils.clip_grad_norm_(model.parameters(), clip)
            opt.step()
            n_update += 1
        vl, va = evaluasi(model, ds_val)
        riwayat['val_loss'].append(vl); riwayat['val_acc'].append(va)

    tl, _ = evaluasi(model, ds_latih)
    durasi = time.perf_counter() - mulai
    return model, riwayat, {
        'run_id': label, 'seed': SEED, 'model': label, 'vocab': V,
        'panjang_maks': MAKS, 'hidden': HID,
        'parameter': sum(p.numel() for p in model.parameters()),
        'n_update': n_update, 'clip': clip if clip else 'tidak',
        'train_loss': tl, 'val_loss': riwayat['val_loss'][-1],
        'val_acc': riwayat['val_acc'][-1],
        'grad_norm_mean': float(np.mean(riwayat['grad_norm'])),
        'detik_per_epoch': durasi / epoch,
    }

hasil, kurva, simpan = [], {}, {}
for model, label, clip in [(RerataEmbedding(), 'rerata-embedding', None),
                           (ModelRNN(False), 'rnn-tanpa-packing', None),
                           (ModelRNN(True), 'rnn-packing', None),
                           (ModelRNN(True), 'rnn-packing-clip', 1.0)]:
    m_, r, catatan = jalankan(model, label, clip)
    hasil.append(catatan); kurva[label] = r; simpan[label] = m_

pd.DataFrame(hasil)[['run_id', 'parameter', 'n_update', 'val_loss', 'val_acc',
                     'grad_norm_mean', 'detik_per_epoch']]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
for label, r in kurva.items():
    ax[0].plot(range(1, EPOCH + 1), r['val_acc'], marker='o', label=label)
ax[0].set_xlabel('epoch'); ax[0].set_ylabel('validation accuracy')
ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)

for label in ('rnn-packing', 'rnn-packing-clip'):
    ax[1].plot(kurva[label]['grad_norm'], lw=0.7, label=label)
ax[1].set_xlabel('update'); ax[1].set_ylabel('norma gradien sebelum dipotong')
ax[1].set_yscale('log'); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

Bandingkan dua baris RNN pada tabel: selisih akurasi antara berpengemasan dan tidak adalah harga yang dibayar untuk satu baris kode yang keliru. Bandingkan pula terhadap `rerata-embedding` — bila selisihnya tipis, tugas ini memang tidak terlalu menuntut informasi urutan.

## 5. Akurasi menurut panjang teks

In [ ]:
juara = max(hasil, key=lambda c: c['val_acc'])['run_id']
model = simpan[juara]

@torch.no_grad()
def prediksi(model, ds):
    model.eval()
    p, t = [], []
    for xb, lb, yb in loader(ds, 256, False):
        p.append(model(xb.to(DEVICE), lb).argmax(1).cpu()); t.append(yb)
    return torch.cat(p), torch.cat(t)

pred, target = prediksi(model, ds_val)
golongan = pd.cut(L_val.numpy(), bins=[0, 20, 40, MAKS],
                  labels=['<20', '20-40', '>40'])
ringkas = (pd.DataFrame({'golongan': golongan, 'benar': (pred == target).numpy()})
           .groupby('golongan', observed=True)['benar']
           .agg(['mean', 'size']).rename(columns={'mean': 'akurasi', 'size': 'jumlah'}))
print(f'model terbaik: {juara}')
print(ringkas.to_string())

## Exit ticket

1. Mengapa `H[:, -1]` berubah ketika bantalan ditambah, sedangkan `h_n` tidak?
2. Berapa persen parameter model berada pada tabel embedding, dan apa yang mengubah angka itu?
3. Pemotongan gradien menyembuhkan gejala yang mana — gradien meledak atau gradien lenyap?

**Tugas setelah sesi:** kerjakan `starter-mahasiswa.ipynb` — empat run inti, dua run panjang maksimum, dua run ukuran keadaan, lalu satu evaluasi test.